In [5]:
import json
import cv2 as cv
from pathlib import Path

# Paths
base_dir = Path.cwd()  # current working directory
data_dir = base_dir / "labeled_example_data"
labels_path = data_dir / "labels.json"
output_dir = base_dir / "labeled_with_red_dot"
output_dir.mkdir(exist_ok=True)

# Load labels
with open(labels_path, "r") as f:
    labels = json.load(f)

# Loop through all entries
for entry in labels:
    img_path = data_dir / entry["filename"]
    img = cv.imread(str(img_path))
    if img is None:
        print(f"⚠️ Could not read {img_path}")
        continue

    h, w = img.shape[:2]

    # Draw red dot if wheel is present
    if entry["is_present"] == 1.0:
        x_px = int(entry["center"][0] * w)
        y_px = int(entry["center"][1] * h)
        cv.circle(img, (x_px, y_px), 5, (0, 0, 255), -1)  # red filled circle

    # Save annotated image
    out_path = output_dir / entry["filename"]
    cv.imwrite(str(out_path), img)

print(f"✅ Annotated images saved to: {output_dir}")

✅ Annotated images saved to: /Users/amrattia/projects/ATI Perception Take Home Challenge/Challenge 1 - Rim Center Location/labeled_with_red_dot


In [1]:
import json
import cv2 as cv
import numpy as np
from pathlib import Path

# --- Paths ---
base_dir = Path.cwd()  # run from project root
data_dir = base_dir / "labeled_example_data"
labels_path = data_dir / "labels.json"
output_dir = base_dir / "debug_check"
output_dir.mkdir(exist_ok=True)

# --- Load labels ---
with open(labels_path, "r") as f:
    labels = json.load(f)
labels_by_name = {e["filename"]: e for e in labels}

# --- Crop helper ---
def crop_around_center(img, cx, cy, box_size=300):
    H, W = img.shape[:2]
    half = box_size // 2
    x0 = max(0, cx - half)
    y0 = max(0, cy - half)
    x1 = min(W, cx + half)
    y1 = min(H, cy + half)
    return img[y0:y1, x0:x1], x0, y0

# --- Utilities for constrained fitting ---
def radial_edge_histogram(edges, cx, cy, r_max=None):
    ys, xs = np.nonzero(edges)
    if len(xs) == 0:
        return None, None
    rs = np.hypot(xs - cx, ys - cy)
    if r_max is None:
        r_max = int(max(edges.shape) * 0.6)
    rs_clipped = rs[(rs > 5) & (rs < r_max)]
    if len(rs_clipped) == 0:
        return None, None
    bins = np.arange(0, r_max + 1)
    hist, _ = np.histogram(rs_clipped, bins=bins)
    peak_idx = np.argmax(hist)
    return peak_idx, hist

def mask_annulus(shape, cx, cy, r, width=8):
    Y, X = np.ogrid[:shape[0], :shape[1]]
    dist = np.sqrt((X - cx)**2 + (Y - cy)**2)
    return (dist >= (r - width)) & (dist <= (r + width))

def fit_ellipse_on_points(points):
    if len(points) < 50:
        return None
    pts = points.astype(np.float32).reshape(-1, 1, 2)
    try:
        return cv.fitEllipse(pts)
    except cv.error:
        return None

def score_ellipse(ell, H, W, cx0, cy0):
    (cx, cy), (MA, ma), _ = ell
    a, b = max(MA, ma), min(MA, ma)
    if a <= 0 or b <= 0:
        return -1
    ar = b / (a + 1e-6)
    area = np.pi * 0.25 * a * b / (H * W)
    center_prox = 1.0 - min(1.0, np.hypot(cx - cx0, cy - cy0) / (0.5 * max(H, W)))
    return 0.5 * ar + 0.3 * area + 0.2 * center_prox

def detect_center_hough(gray, r_hint=None):
    H, W = gray.shape[:2]
    blur = cv.GaussianBlur(gray, (9, 9), 2)
    minR = 10
    maxR = int(0.6 * max(H, W))
    if r_hint is not None:
        minR = max(10, int(0.7 * r_hint))
        maxR = min(maxR, int(1.3 * r_hint))
    circles = cv.HoughCircles(
        blur, cv.HOUGH_GRADIENT, dp=1.2, minDist=int(0.4 * min(H, W)),
        param1=120, param2=25, minRadius=minR, maxRadius=maxR
    )
    if circles is not None and len(circles) > 0:
        x, y, r = circles[0][0]
        return int(round(x)), int(round(y))
    return None

def detect_center_annulus(img_crop):
    gray = cv.cvtColor(img_crop, cv.COLOR_BGR2GRAY)
    gray = cv.equalizeHist(gray)
    gray_blur = cv.GaussianBlur(gray, (5, 5), 0)
    edges = cv.Canny(gray_blur, 40, 120)

    H, W = gray.shape[:2]
    cx0, cy0 = W // 2, H // 2

    r_peak, _ = radial_edge_histogram(edges, cx0, cy0)
    if r_peak is None:
        return detect_center_hough(gray)

    ann = mask_annulus(edges.shape, cx0, cy0, r_peak, width=10)
    ann_edges = (edges > 0) & ann
    ys, xs = np.nonzero(ann_edges)
    if len(xs) < 50:
        ann = mask_annulus(edges.shape, cx0, cy0, r_peak, width=16)
        ann_edges = (edges > 0) & ann
        ys, xs = np.nonzero(ann_edges)

    if len(xs) >= 50:
        ell = fit_ellipse_on_points(np.column_stack([xs, ys]))
        if ell is not None:
            s = score_ellipse(ell, H, W, cx0, cy0)
            if s > 0:
                (ex, ey), _, _ = ell
                return int(round(ex)), int(round(ey))

    return detect_center_hough(gray, r_hint=r_peak)

# --- Batch loop ---
errors = []
fail = []

for fname, entry in labels_by_name.items():
    img_path = data_dir / fname
    img = cv.imread(str(img_path))
    if img is None:
        print(f"⚠️ Missing image: {fname}")
        continue

    H, W = img.shape[:2]
    if entry["is_present"] != 1.0:
        continue

    # Ground truth pixel coords
    gt_x = int(round(entry["center"][0] * W))
    gt_y = int(round(entry["center"][1] * H))

    # Crop around GT
    crop_img, x0, y0 = crop_around_center(img, gt_x, gt_y, box_size=320)

    # Detect center
    det_crop = detect_center_annulus(crop_img)

    # Draw GT (red)
    cv.circle(img, (gt_x, gt_y), 5, (0, 0, 255), -1)
    cv.drawMarker(img, (gt_x, gt_y), (0, 0, 255),
                  markerType=cv.MARKER_CROSS, markerSize=16, thickness=2)

    if det_crop is not None:
        gx_crop, gy_crop = det_crop
        gx_full, gy_full = gx_crop + x0, gy_crop + y0
        cv.circle(img, (gx_full, gy_full), 6, (0, 255, 0), 2)
        cv.drawMarker(img, (gx_full, gy_full), (0, 255, 0),
                      markerType=cv.MARKER_TILTED_CROSS, markerSize=20, thickness=2)
        err = float(np.hypot(gx_full - gt_x, gy_full - gt_y))
        errors.append(err)
    else:
        fail.append(fname)

    # Save overlay
    cv.imwrite(str(output_dir / fname), img)

# --- Summary ---
if errors:
    print(f"✅ Processed {len(errors)} detections")
    print(f"📏 Average pixel error: {np.mean(errors):.2f}px | Median: {np.median(errors):.2f}px")
if fail:
    print(f"⚠️ Failed detections: {len(fail)} (e.g., {fail[:5]})")
print(f"🖼️ Overlays saved to: {output_dir}")

✅ Processed 1000 detections
📏 Average pixel error: 32.33px | Median: 31.60px
🖼️ Overlays saved to: /Users/amrattia/projects/ATI Perception Take Home Challenge/Challenge 1 - Rim Center Location/debug_check
